# 14 · Explainable ML: permutation importance, PDP, ICE y SHAP

Explicar un modelo puede significar cosas distintas: comportamiento global, razón de una predicción, sensibilidad a una variable o evidencia causal. **Explainability no equivale a causalidad.**

## Objetivos
- Diferenciar explicación global y local.
- Comparar impurity importance y permutation importance.
- Usar Partial Dependence (PDP) e ICE.
- Introducir SHAP y valores de Shapley.
- Reconocer problemas con features correlacionadas y proxies.
- Diseñar explicaciones útiles para auditoría y negocio.


In [ ]:
!pip -q install shap
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import roc_auc_score
import shap
SEED=42
X,y=load_breast_cancer(return_X_y=True,as_frame=True)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEED)
model=RandomForestClassifier(n_estimators=500,min_samples_leaf=3,random_state=SEED,n_jobs=-1).fit(Xtr,ytr)
print('AUC',roc_auc_score(yte,model.predict_proba(Xte)[:,1]))

## 1. Importancia basada en impureza
Random Forest acumula cuánto reducen impureza los splits de cada feature. Es rápida, pero puede favorecer variables continuas o de alta cardinalidad y repartir importancia de forma extraña entre variables correlacionadas.


In [ ]:
imp=pd.Series(model.feature_importances_,index=X.columns).sort_values(ascending=False); imp.head(12).sort_values().plot.barh(title='Impurity importance'); plt.show()

## 2. Permutation Importance
Permuta una feature y mide cuánto empeora la métrica. Si el modelo puede reemplazar información con una feature correlacionada, la caída puede ser pequeña aunque ambas sean importantes en conjunto. Por eso la interpretación depende del conjunto de features.


In [ ]:
perm=permutation_importance(model,Xte,yte,scoring='roc_auc',n_repeats=20,random_state=SEED,n_jobs=-1)
pd.Series(perm.importances_mean,index=X.columns).sort_values(ascending=False).head(12).sort_values().plot.barh(title='Permutation importance'); plt.show()

## 3. Partial Dependence (PDP)
PDP pregunta: 'en promedio, ¿cómo cambia la predicción cuando fijamos una feature en distintos valores?'. Puede evaluar combinaciones que casi no existen en los datos si las features están correlacionadas.

ICE dibuja una curva por observación y permite detectar heterogeneidad que un promedio PDP oculta.


In [ ]:
top=imp.index[:3].tolist(); fig,ax=plt.subplots(1,3,figsize=(15,4)); PartialDependenceDisplay.from_estimator(model,Xte,top,kind='both',subsample=80,random_state=SEED,ax=ax); plt.tight_layout(); plt.show()

## 4. SHAP y valores de Shapley
SHAP adapta una idea de teoría de juegos: repartir la diferencia entre una predicción y un valor base entre las features. Para árboles, TreeSHAP es eficiente.

Una explicación local se expresa aproximadamente como:
$$f(x)=E[f(X)]+\sum_j \phi_j$$

Los $\phi_j$ dependen del modelo, background y supuestos sobre dependencia de features. No significan 'si cambiara esta variable, causaría este efecto'.


In [ ]:
explainer=shap.TreeExplainer(model)
sv=explainer(Xte.iloc[:200])
# Para clasificadores, SHAP puede devolver una dimensión por clase según versión.
print('shape SHAP',sv.values.shape)
try:
    shap.plots.beeswarm(sv[:,:,1] if sv.values.ndim==3 else sv,max_display=15)
except Exception as e:
    print('La API de SHAP varía entre versiones:',e)

## 5. Explicación local
Para una predicción individual interesa entender qué variables empujaron score hacia arriba/abajo. Esto es útil en revisión humana, debugging y auditoría. Pero una explicación no debe mostrarse como justificación normativa si el modelo usa proxies, datos con sesgo o features no accionables.


In [ ]:
i=0; print('probabilidad clase 1=',model.predict_proba(Xte.iloc[[i]])[0,1])
try:
    one=sv[i,:,1] if sv.values.ndim==3 else sv[i]
    shap.plots.waterfall(one,max_display=12)
except Exception as e: print(e)

## 6. Correlación y explicaciones condicionales
Si ingreso y ocupación, o edad y antigüedad, están fuertemente relacionados, 'permutar' una sola feature crea combinaciones irreales. Métodos condicionales, agrupación de features y conocimiento de dominio pueden ser necesarios.

## 7. Counterfactual explanations
Preguntan qué cambio mínimo haría variar la decisión. Deben respetar restricciones: edad no debe 'cambiar', variables relacionadas deben mantenerse coherentes y no se deben sugerir acciones imposibles o discriminatorias. Herramientas: DiCE, Alibi.

## Responsible use
- documenta uso previsto;
- separa explicación de causalidad;
- audita proxies de variables sensibles;
- registra versión del modelo;
- mide estabilidad de explicaciones;
- proporciona contexto e incertidumbre.

## Ejercicios
1. Compara impurity vs permutation importance con dos features correlacionadas.
2. Genera PDP e ICE para una feature con interacción.
3. Calcula SHAP global y local para XGBoost.
4. Crea un counterfactual con restricciones realistas.
5. Compara explicaciones del mismo caso en dos versiones del modelo.
6. Diseña una ficha de explicación para un usuario no técnico.
7. Investiga ALE plots, LIME y Anchors.
